##### Copyright 2026 Google LLC.

In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Gemini API: Code Execution

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/Code_Execution.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>


The Gemini API [code execution](https://ai.google.dev/gemini-api/docs/code-execution) feature enables the model to generate and run Python code based on plain-text instructions that you give it, and even output graphs. It can learn iteratively from the results until it arrives at a final output.

This notebook is a walk through:
* Understanding how to start using the code execution feature with Gemini API
* Learning how to use code execution on single Gemini API calls
* Running scenarios using local files (or files uploaded to the Gemini File API) via File I/O
* Using code execution on chat interactions
* Performing code execution on multimodal scenarios

## Setup

### Install SDK

Install the SDK from [PyPI](https://github.com/googleapis/python-genai).

In [ ]:
%pip install -q -U "google-genai>=1.0.0"

### Setup your API key

To run the following cell, your API key must be stored it in a Colab Secret named `GOOGLE_API_KEY`. If you don't already have an API key, or you're not sure how to create a Colab Secret, see [Authentication ![image](https://storage.googleapis.com/generativeai-downloads/images/colab_icon16.png)](../quickstarts/Authentication.ipynb) for an example.

In [ ]:
from google.colab import userdata

GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')

### Initialize SDK client

With the new SDK you now only need to initialize a client with you API key (or OAuth if using [Vertex AI](https://cloud.google.com/vertex-ai)). The model is now set in each call.

In [ ]:
from google import genai

client = genai.Client(api_key=GOOGLE_API_KEY)

### Choose a model

Now select the model you want to use in this guide, either by selecting one in the list or writing it down. Keep in mind that some models, like the 2.5 ones are thinking models and thus take slightly more time to respond (cf. [thinking notebook](./Get_started_thinking.ipynb) for more details and in particular learn how to switch the thiking off).

For more information about all Gemini models, check the [documentation](https://ai.google.dev/gemini-api/docs/models/gemini) for extended information on each of them.

In [ ]:
MODEL_ID = "gemini-3-flash-preview" # @param ["gemini-2.5-flash-lite", "gemini-2.5-flash", "gemini-2.5-pro", "gemini-2.5-flash-preview", "gemini-3.1-flash-lite-preview", "gemini-3.1-pro-preview"] {"allow-input":true, isTemplate: true}

## Helper function

When using code execution as a tool, the model returns a list of parts including `text`, `executable_code`, `execution_result`, and `inline_data` parts. Use the function below to help you visualize and better display the code execution results. Here are a few details about the different fields of the results:

* `text`: Inline text generated by the model.
* `executable_code`: Code generated by the model that is meant to be executed.
* `code_execution_result`: Result of the `executable_code`.
* `inline_data`: Inline media generated by the model.

In [ ]:
from IPython.display import Image, Markdown, Code, HTML

def display_code_execution_result(response):
  for part in response.candidates[0].content.parts:
    if part.text is not None:
      display(Markdown(part.text))
    if part.executable_code is not None:
      code_html = f'<pre style="background-color: green;">{part.executable_code.code}</pre>' # Change code color
      display(HTML(code_html))
    if part.code_execution_result is not None:
      display(Markdown(part.code_execution_result.output))
    if part.inline_data is not None:
      display(Image(data=part.inline_data.data, width=800, format="png"))
    display(Markdown("---"))

## Use `code_execution` with a single call

When initiating the model, pass `code_execution` as a `tool` to tell the model that it is allowed to generate and run code.

In [ ]:
from google.genai import types

prompt = """
    What is the sum of the first 50 prime numbers?
    Generate and run code for the calculation, and make sure you get all 50.
"""

response = client.models.generate_content(
    model=MODEL_ID,
    contents=prompt,
    config=types.GenerateContentConfig(
        tools=[types.Tool(
            code_execution=types.ToolCodeExecution
            )]
        )
    )

display_code_execution_result(response)

##The Market Lead

The Challenge: You've been assigned to find "Fixer-Upper" opportunities in California. Your first task is to use Code Execution to find the index of a specific bargain block.

Goal: Find a block that is relatively old (high HouseAge) but has a low median value (MedHouseVal).

The dataset you will use in this guide comes from the [StatLib](http://lib.stat.cmu.edu/datasets/) from the [Department of Statistics](https://www.cmu.edu/dietrich/statistics-datascience/index.html) at [Carnegie Mellon University](http://www.cmu.edu/). It is made available by the [`scikit-learn`](https://scikit-learn.org) under the 3-Clause BSD license.

It provides 20k information on various blocks in Californina, including the location (longitute/lattitude), average income,
housing average age, average rooms, average bedrooms, population,
average occupation.

Here's a breakdown of the columns and what the attributes represent:
* MedInc:        median income in block group
* HouseAge:      median house age in block group
* AveRooms:      average number of rooms per household
* AveBedrms:     average number of bedrooms per household
* Population:    block group population
* AveOccup:      average number of household members
* Latitude:      block group latitude
* Longitude:     block group longitude

**Note**: Code execution functionality works best with a `.csv` or `.txt` file.


In [ ]:
import pandas as pd
from sklearn.datasets import fetch_california_housing

california_housing = fetch_california_housing(as_frame=True)
california_housing.frame.to_csv('houses.csv', index=False)

In [ ]:
# Read the CSV file into a pandas DataFrame
houses_data = pd.read_csv('houses.csv', nrows=5000) # only keeping the first 5000 entries to keep the request light (still 500k tokens). Use pro models to ingest the full dataset.
houses_data.to_csv('houses.csv', index=False)
houses_data.head()

In [ ]:
from google.genai import types

# Upload diving_data.csv file using the File API
houses_file = client.files.upload(
    file='houses.csv',
    config=types.FileDict(display_name='Blocks Data')
)

print(f"Uploaded file '{houses_file.display_name}' as: {houses_file.uri}")

In [ ]:
# PROJECT TASK 1: Find your unique lead
# Instruction: Ask Gemini to find 3 different blocks that meet your
# personal criteria (e.g., "HouseAge > 40 and MedInc < 3.0").
# Pick ONE index to be your property for the final report.

user_prompt = "find 3 different blocks that have a HouseAge > 40, MedInc < 3.0, and Population > 300"

response = client.models.generate_content(
    model=MODEL_ID,
    contents=[user_prompt, houses_file],
    config=types.GenerateContentConfig(
        tools=[types.Tool(code_execution=types.ToolCodeExecution)]
    )
)

display_code_execution_result(response)

##Activity 2: Visual Intelligence & Anomaly Detection

Data isn't just numbers; it's patterns. In this step, we ask the model to generate Scatterplots and Violin Plots to visualize price variance and identify anomalies. This shows the model's ability to use libraries like matplotlib and seaborn to provide a deeper investment analysis.

In [ ]:
response = client.models.generate_content(
    model=MODEL_ID,
    contents=[
        "This dataset provides information on various blocks in Californina.",
        "Generate a scatterplot comparing the houses age with the median house value for the top-20 most expensive blocks.",
        "Use each black as a different color, and include a legend of what each color represents.",
        "Plot the age as the x-axis, and the median house value as the y-axis.",
        "In addition, point out on the graph which points could be anomalies? Circle the anomaly in red on the graph."
        "Then save the plot as an image file and display the image.",
        houses_file
    ],
    config=types.GenerateContentConfig(
        tools=[types.Tool(code_execution=types.ToolCodeExecution)]
    )
)

display_code_execution_result(response)

Moving forward with the data investigation, you can now analyze data variance in the dataset:

In [ ]:
response = client.models.generate_content(
    model=MODEL_ID,
    contents=[
        "This dataset provides information on various blocks in Californina.",
        "Calculate the variance of the house price for houses between 15 and 25 Years old",
        "Plot the variance using a violinplot",
        "I would like you to use the x-axis for the house age, and house price for the y-axis",
        "Then save the plot as an image file and display the image.",
        houses_file
    ],
    config=types.GenerateContentConfig(
        tools=[types.Tool(code_execution=types.ToolCodeExecution)]
    )
)

display_code_execution_result(response)

Here is another example - Calculating repeated letters in a word (a common example where LLM sometimes struggle to get the result).

In [ ]:
response = client.models.generate_content(
    model=MODEL_ID,
    contents="Calculate how many letter r in the word strawberry and show the code used to do it",
    config=types.GenerateContentConfig(
        tools=[types.Tool(code_execution=types.ToolCodeExecution)]
    )
)

In [ ]:
display_code_execution_result(response)

##Intermission: Natural Language & Logic

Before we move to Computer Vision, let's see how Gemini handles classic LLM pitfalls—like counting letters or explaining inefficient algorithms—using real-time Python execution.**bold text**

It works the same when using a `chat`, which allows you to have multi-turn conversations with the model. You can set the `system_instructions` as well, which allows you to further steer the behavior of the model.

In [ ]:
system_instruction = """
  You are an expert software developer and a helpful coding assistant.
  You are able to generate high-quality code in any programming language.
"""

chat = client.chats.create(
    model=MODEL_ID,
    config=types.GenerateContentConfig(
        system_instruction=system_instruction,
        tools=[types.Tool(code_execution=types.ToolCodeExecution)],
    ),
)

This time, you're going to ask the model to use a [Bogo-sort](https://en.wikipedia.org/wiki/Bogosort) algorithm to sort a list of numbers.

In [ ]:
response = chat.send_message("Run the bogo-sort algorithm with this list of numbers as input until it is sorted: [2,34,1,65,4]")
display_code_execution_result(response)

This code seems satisfactory, as it performs the task. However, you can further update the code by sending the following message below the model so that it can mitigate some of the randomness.

In [ ]:
response = chat.send_message("Run an alternate implementation of the bogo-sort algorithm with the same input")
display_code_execution_result(response)

response = chat.send_message("How many iterations did it take this time? Compare it with the first try.")
display_code_execution_result(response)

Try running the previous cell multiple times and you'll see a different number of iterations, indicating that the Gemini API indeed ran the code and obtained different results due to the nature of the algorithm.

##The Forensic Walkthrough (Multimodal Vision)



The Challenge: We’ve identified a potential lead in the messy kitchen sink. You need to prove this house is a "distressed asset" to justify a lower offer.

Goal: Use High-Resolution Vision to find 3 specific items in the sink or on the counter that prove the house has been neglected.

In [ ]:
import requests
import PIL.Image
from io import BytesIO

# The URL for the messy kitchen image you selected
img_url = "https://upload.wikimedia.org/wikipedia/commons/6/68/Messy_kitchen_sink.jpg"

# The "Secret Sauce": This header makes the request look like it's from a browser
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}

try:
    # Send the request with the headers included
    response = requests.get(img_url, headers=headers)
    response.raise_for_status()

    # Load and save the image
    property_image = PIL.Image.open(BytesIO(response.content))
    property_image.save("kitchen_clutter.jpg")

    display(property_image)
    print("Slide 16 SUCCESS: Messy Kitchen Image Loaded!")
except Exception as e:
    print(f"Error: {e}")

In [ ]:
# PROJECT TASK 2: High-Resolution Inspection
# Instructions: Ask Gemini to find 3 signs of property neglect.
# Challenge: Can you find a detail that the ai can't answer on "MEDIA_RESOLUTION_LOW"?
from google.genai import types

inspection_prompt = "What is the status of this property basedo on the kitchen. Are there 3 signs of property neglect?"

response = client.models.generate_content(
    model=MODEL_ID,
    contents=[inspection_prompt, property_image],
    config=types.GenerateContentConfig(
        media_resolution="MEDIA_RESOLUTION_HIGH"
    )
)

print(response.text)

##Bonus: Visual Logic & Probability

To demonstrate Gemini's ability to combine Computer Vision with Mathematical Logic, we use the classic Monty Hall problem. The model must 'look' at the diagram of the doors, understand the spatial context, and then write and execute Python code to simulate the probability of winning.

In [ ]:
! curl -o montey_hall.png https://upload.wikimedia.org/wikipedia/commons/thumb/3/3f/Monty_open_door.svg/640px-Monty_open_door.svg.png

In [ ]:
import PIL
montey_hall_image = PIL.Image.open("montey_hall.png")
montey_hall_image

In [ ]:
prompt="""
    Run a simulation of the Monty Hall Problem with 1,000 trials.

    The answer has always been a little difficult for me to understand when people
    solve it with math - so run a simulation with Python to show me what the
    best strategy is.
"""
result = client.models.generate_content(
    model=MODEL_ID,
    contents=[
        prompt,
        montey_hall_image
    ],
    config=types.GenerateContentConfig(
        tools=[types.Tool(code_execution=types.ToolCodeExecution)]
    )
)

display_code_execution_result(result)

## Streaming

Streaming is compatible with code execution, and you can use it to deliver a response in real time as it gets generated. Just note that successive parts of the same type (`text`, `executable_code` or `execution_result`) are meant to be joined together, and you have to stitch the output together yourself:

In [ ]:
# Streaming a Live Market Analysis
prompt = """
Give me a live, blow-by-blow comparison of Block 89 versus the
top 5 most expensive blocks we found earlier.
Stream your thoughts on whether this is a 'Value Trap' or a 'Hidden Gem.'
"""

result = client.models.generate_content_stream(
    model=MODEL_ID,
    contents=[prompt, houses_file, property_image],
    config=types.GenerateContentConfig(
        tools=[types.Tool(code_execution=types.ToolCodeExecution)]
    )
)

for chunk in result:
    display_code_execution_result(chunk)

##The Final Investment Thesis
Combine your market data from Slide 15 and your visual findings from Slide 16. You must convince the Investment Committee to either BUY this house for a flip or PASS.

Requirement: Your prompt must reference both the houses_file and the property_image.

In [ ]:
# SLIDE 17: Build Your Own Analyst
# Instructions: Change the values below to tune your AI's behavior.

# 1. RISK_APPETITE: Choose 'High' (Flippers) or 'Low' (Safe Investors)
RISK_APPETITE = "High"

# 2. FOCUS_AREA: What do you care about most?
# Options: "Market Data", "Visual Condition", or "Renovation Potential"
FOCUS_AREA = "Visual Condition"

# 3. PITCH_TONE: How should the AI talk to the board?
# Options: "Aggressive", "Cautious", or "Visionary"
PITCH_TONE = "Visionary"

# The prompt now uses the students' specific choices
project_prompt = f"""
Act as a Senior Analyst with a {RISK_APPETITE} risk appetite.
Write a 2-paragraph investment pitch for Block (INSERT YOUR BLOCK INDEX HERE!!!) in a {PITCH_TONE} tone.
Focus primarily on the {FOCUS_AREA} found in the houses_file and property_image.
"""

response = client.models.generate_content(
    model=MODEL_ID,
    contents=[project_prompt, houses_file, property_image]
)

print(f"--- {PITCH_TONE} PITCH (Focus: {FOCUS_AREA}) ---")
print(response.text)